In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

In [8]:
from app.agents.orchestrator import network_agent
from ragas.integrations.langgraph import convert_to_ragas_messages
from ragas.dataset_schema import MultiTurnSample
from ragas import EvaluationDataset
from ragas.metrics import _AgentGoalAccuracyWithReference, _ToolCallAccuracy, _TopicAdherenceScore


In [30]:
test_cases = [
    {
        "ticket_number": "TKT-2025001",
        "user_input": "Analyse ticket TKT-2025001",
        "reference": "The agent successfully retrieved ticket TKT-2025001, identified the affected node, retrieved the full inventory path, checked planned changes, diagnosed the root cause using the network guide, and provided structured resolution steps with SLA status.",
        "reference_topics": ["network fault analysis", "inventory lookup", "planned changes", "root cause analysis", "SLA"],
    },
    {
        "ticket_number": "TKT-2025010",
        "user_input": "Analyse ticket TKT-2025010",
        "reference": "The agent successfully retrieved ticket TKT-2025010, identified the affected node, retrieved the full inventory path, checked planned changes, diagnosed the root cause using the network guide, and provided structured resolution steps with SLA status.",
        "reference_topics": ["network fault analysis", "inventory lookup", "planned changes", "root cause analysis", "SLA"],
    },
    {
        "ticket_number": "TKT-9999999",
        "user_input": "Analyse ticket TKT-9999999",
        "reference": "The agent attempted to fetch ticket TKT-9999999 and correctly returned a not found message without hallucinating ticket details or producing a false analysis.",
        "reference_topics": ["ticket lookup", "error handling"],
    },
]

In [31]:
import uuid
from langchain_core.messages import HumanMessage

samples = []

for case in test_cases:
    thread_id = f"eval-{case['ticket_number']}-{uuid.uuid4().hex[:8]}"
    
    result = network_agent.invoke(
        {
            "messages": [HumanMessage(content=case["user_input"])],
            "ticket_number": case["ticket_number"],
            "ticket_details": "",
            "inventory_path": "",
            "correlation_findings": "",
            "rca_findings": "",
            "changes_found": False,
            "next": "",
        },
        config={"configurable": {"thread_id": thread_id}},
    )
    
    ragas_messages = convert_to_ragas_messages(result["messages"])
    sample = MultiTurnSample(
        user_input=ragas_messages,
        reference=case["reference"],
        reference_topics=case["reference_topics"],
    )
    samples.append(sample)
    print(f"✓ {case['ticket_number']} processed")

agent_eval_dataset = EvaluationDataset(samples=samples)

[Orchestrator] Fetching ticket: TKT-2025001
[Orchestrator] Ticket fetched — passing to Correlation Agent.
[Correlation Agent] Checking inventory and planned changes...
[Correlation Agent] Complete. Changes found: False
[RCA Agent] No changes found — performing technical RCA...
[RCA Agent] Complete.
[Network Agent] Building RCA summary...
[Network Agent] Summary complete.
✓ TKT-2025001 processed
[Orchestrator] Fetching ticket: TKT-2025010
[Orchestrator] Ticket fetched — passing to Correlation Agent.
[Correlation Agent] Checking inventory and planned changes...
[Correlation Agent] Complete. Changes found: False
[RCA Agent] No changes found — performing technical RCA...
[RCA Agent] Complete.
[Network Agent] Building RCA summary...
[Network Agent] Summary complete.
✓ TKT-2025010 processed
[Orchestrator] Fetching ticket: TKT-9999999
[Orchestrator] Ticket not found: TKT-9999999
[Orchestrator] Ticket fetched — passing to Correlation Agent.
[Correlation Agent] Checking inventory and planned ch

In [32]:
from ragas import evaluate, RunConfig
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

agent_result = evaluate(
    dataset=agent_eval_dataset,
    metrics=[
        _AgentGoalAccuracyWithReference(),
        _TopicAdherenceScore(),
    ],
    llm=evaluator_llm,
    run_config=RunConfig(timeout=360),
)
agent_result

/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_12803/3422410842.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
Evaluating: 100%|██████████| 6/6 [00:31<00:00,  5.30s/it]


{'agent_goal_accuracy': 0.3333, 'topic_adherence(mode=f1)': 0.6004}

In [33]:
agent_result.to_pandas()[['agent_goal_accuracy', 'topic_adherence(mode=f1)']]

,agent_goal_accuracy,topic_adherence(mode=f1)
0,0.0,0.416667
1,1.0,0.769231
2,0.0,0.615385


In [34]:
for i, sample in enumerate(samples):
    print(f"\n=== Sample {i} ===")
    print(sample.user_input[-1].content[:600])
    print("---")


=== Sample 0 ===
TICKET  
  Number   : TKT-2025001  
  Priority : H  
  Node     : DSLAM-EDI-028  
  Issue    : Metro Bridge node CPU utilisation exceeded 90%, potential memory leak in routing process  

SUMMARY  
  The CPU utilization on the DSLAM-EDI-028 node has exceeded 90%, indicating a potential memory leak in the routing process. This high utilization may be caused by a broadcast storm or misconfiguration in rate-limiting settings.

ROOT CAUSE  
  High CPU utilization (>90%) on the DSLAM node may indicate a possible broadcast storm or misconfiguration in rate-limiting settings.

EVIDENCE  
  - High CPU 
---

=== Sample 1 ===
TICKET  
  Number   : TKT-2025010  
  Priority : H  
  Node     : CPE-EDI-091  
  Issue    : Peta Core BGP peer flap, route instability affecting 12 MC nodes  

SUMMARY  
  The incident involves a BGP peer flap at the Peta Core, leading to route instability that is impacting 12 MC nodes. Initial analysis indicates potential physical layer issues or misconfi